# EOR GRF Sky Model Diagnostics

Sanity-check notebook for the EOR Gaussian Random Field sky model realization.

**Sky model path:** `eor-grf-256/rlzn_seed_111_offsetfix`  
**Format:** `.skyh5` (pyradiosky HEALPix), one file per frequency channel  
**Parameters:** nside=256, RING ordering, ICRS frame, 89 channels (fch0227–fch0315)  
**Freq range:** 74.6–85.4 MHz (z ~ 15.6–18.0), channel spacing 122.07 kHz  
**GRF input:** P(k) ~ k^{-2.7}, Planck15 cosmology, ell_max=1250

### Tests
1. File metadata & consistency checks
2. Pixel ordering validation (RING vs NESTED)
3. Mollweide sky maps at multiple frequencies
4. Pixel value histogram & Gaussianity
5. Angular power spectrum C_ell
6. Frequency spectra (per-pixel)
7. Cross-frequency correlation matrix
8. Delay spectrum / line-of-sight P(k_parallel)
9. Variance vs frequency

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import h5py
from pathlib import Path
from astropy import units, cosmology
from scipy import stats

%matplotlib inline
plt.rcParams.update({"figure.figsize": (12, 5), "font.size": 12})

SKY_DIR = Path("/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim/sky_models/eor-grf-256/rlzn_seed_556_offsetfix/")
FILES = sorted(SKY_DIR.glob("fch*.skyh5"))
print(f"Found {len(FILES)} channel files: {FILES[0].name} ... {FILES[-1].name}")

## 1. File Metadata & Consistency Checks

Verify that all 89 channel files have consistent nside, ordering, pixel indices, and expected frequency spacing.

In [ ]:
# Scan all files and collect metadata
freqs_hz = []
nsides = []
orderings = []
stokes_shapes = []
hpx_inds_ok = []  # True if contiguous 0..npix-1

for f in FILES:
    with h5py.File(f, "r") as fl:
        hdr = fl["Header"]
        freqs_hz.append(hdr["freq_array"][0])
        nsides.append(int(hdr["nside"][()]))
        orderings.append(hdr["hpx_order"][()].decode())
        stokes_shapes.append(fl["Data"]["stokes"].shape)
        inds = hdr["hpx_inds"][:]
        hpx_inds_ok.append(np.array_equal(inds, np.arange(len(inds))))

freqs_hz = np.array(freqs_hz)
freqs_mhz = freqs_hz / 1e6

# Print summary
print(f"Nside values:  {set(nsides)}")
print(f"Orderings:     {set(orderings)}")
print(f"Stokes shapes: {set(stokes_shapes)}")
print(f"All hpx_inds contiguous 0..npix-1: {all(hpx_inds_ok)}")
print(f"Freq range:    {freqs_mhz[0]:.4f} – {freqs_mhz[-1]:.4f} MHz")
print(f"Channel spacing: {np.unique(np.diff(freqs_hz)/1e3)} kHz")
print(f"Total bandwidth: {(freqs_mhz[-1] - freqs_mhz[0]):.4f} MHz")

# Redshift range (21cm line at 1420.405 MHz)
z_arr = 1420.405e6 / freqs_hz - 1
print(f"Redshift range: z = {z_arr[-1]:.3f} – {z_arr[0]:.3f}")

## 2. Pixel Ordering Validation

Display the same map interpreted as RING (correct) and as NESTED (incorrect).  
If RING is correct, the RING plot shows smooth spatial structure; the NESTED plot looks scrambled.

In [ ]:
# Load a single representative map (middle channel)
mid_idx = len(FILES) // 2
with h5py.File(FILES[mid_idx], "r") as fl:
    test_map = fl["Data"]["stokes"][0, 0, :]
    test_freq = fl["Header"]["freq_array"][0]

print(f"Test channel: {FILES[mid_idx].name}, freq = {test_freq/1e6:.2f} MHz")

fig = plt.figure(figsize=(16, 5))

# Interpreted as RING (should look correct)
hp.mollview(test_map, nest=False, title="Interpreted as RING (correct)",
            unit="Jy/sr", fig=fig.number, sub=121, cmap="inferno")

# Interpreted as NESTED (should look scrambled if data is actually RING)
hp.mollview(test_map, nest=True, title="Interpreted as NESTED (wrong)",
            unit="Jy/sr", fig=fig.number, sub=122, cmap="inferno")

plt.show()

# Quantitative check: compare spatial smoothness via neighbor variance
nside = hp.npix2nside(len(test_map))
neighbor_var_ring = []
neighbor_var_nest = []
test_pixels = np.random.choice(len(test_map), 5000, replace=False)

for pix in test_pixels:
    neighbors = hp.get_all_neighbours(nside, pix, nest=False)
    neighbors = neighbors[neighbors >= 0]
    neighbor_var_ring.append(np.var(test_map[neighbors]))

# If we reorder to NESTED and treat as RING, neighbors would be wrong
reordered = hp.reorder(test_map, r2n=True)
for pix in test_pixels:
    neighbors = hp.get_all_neighbours(nside, pix, nest=False)
    neighbors = neighbors[neighbors >= 0]
    neighbor_var_nest.append(np.var(reordered[neighbors]))

print(f"\nNeighbor variance (RING, correct):   {np.mean(neighbor_var_ring):.6e}")
print(f"Neighbor variance (wrong ordering):  {np.mean(neighbor_var_nest):.6e}")
print(f"Ratio (should be > 1 if RING is correct): {np.mean(neighbor_var_nest)/np.mean(neighbor_var_ring):.1f}x")

## 3. Mollweide Sky Maps at Multiple Frequencies

Visual inspection at low, mid, and high frequency channels. Maps should show spatially correlated structure (not noise), and the pattern should evolve smoothly across frequency.

In [ ]:
# Show maps at 3 frequencies: low, mid, high
indices = [0, len(FILES)//2, len(FILES)-1]
fig = plt.figure(figsize=(18, 4))

for i, idx in enumerate(indices):
    with h5py.File(FILES[idx], "r") as fl:
        sky = fl["Data"]["stokes"][0, 0, :]
        freq = fl["Header"]["freq_array"][0] / 1e6
    hp.mollview(sky, nest=False, title=f"{FILES[idx].name}\n{freq:.2f} MHz (z={1420.405/freq - 1:.2f})",
                unit="Jy/sr", fig=fig.number, sub=(1, 3, i+1), cmap="inferno",
                min=np.percentile(sky, 1), max=np.percentile(sky, 99))

plt.show()

# Difference map: high freq - low freq (should show structure if signal decorrelates)
with h5py.File(FILES[0], "r") as fl:
    map_lo = fl["Data"]["stokes"][0, 0, :]
with h5py.File(FILES[-1], "r") as fl:
    map_hi = fl["Data"]["stokes"][0, 0, :]

diff = map_hi - map_lo
hp.mollview(diff, nest=False, title=f"Difference: {FILES[-1].name} - {FILES[0].name}",
            unit="Jy/sr", cmap="RdBu_r", min=-np.percentile(np.abs(diff), 99),
            max=np.percentile(np.abs(diff), 99))
plt.show()

## 3b. Angular Scale Filtering — Low / Mid / High ell Modes

Decompose the sky map into spherical harmonics and reconstruct using only selected ell bands. This isolates large-scale (low ell), intermediate, and small-scale (high ell) structure, and verifies that the GRF has the expected spatial content at each scale. The GRF was generated with ell_max=1250, so power above that should be negligible.

In [ ]:
# Filter sky map into ell bands using spherical harmonic decomposition
# Do this for 3 frequencies: low, mid, high
nside = 256
lmax_decomp = 3 * nside - 1  # 767

# Define granular ell bands
bands = [
    (2, 10),
    (10, 30),
    (30, 60),
    (60, 120),
    (120, 200),
    (200, 350),
    (350, 500),
    (500, 767),
]

freq_indices = [0, len(FILES)//2, len(FILES)-1]

for fi in freq_indices:
    with h5py.File(FILES[fi], "r") as fl:
        sky_map = fl["Data"]["stokes"][0, 0, :]
        freq = fl["Header"]["freq_array"][0] / 1e6

    alm = hp.map2alm(sky_map, lmax=lmax_decomp)

    nrows, ncols = 2, 4
    fig = plt.figure(figsize=(22, 10))

    for i, (ell_lo, ell_hi) in enumerate(bands):
        alm_filt = alm.copy()
        for ell in range(lmax_decomp + 1):
            if ell < ell_lo or ell > ell_hi:
                for m in range(ell + 1):
                    idx = hp.Alm.getidx(lmax_decomp, ell, m)
                    alm_filt[idx] = 0.0

        filtered_map = hp.alm2map(alm_filt, nside, lmax=lmax_decomp)

        ell_center = (ell_lo + ell_hi) / 2
        theta_deg = 180.0 / ell_center

        hp.mollview(filtered_map, fig=fig.number, sub=(nrows, ncols, i + 1),
                    title=fr"$\ell$ = {ell_lo}–{ell_hi}  ($\theta \sim$ {theta_deg:.1f}°)",
                    unit="Jy/sr", cmap="RdBu_r",
                    min=-np.percentile(np.abs(filtered_map), 99),
                    max=np.percentile(np.abs(filtered_map), 99))

    plt.suptitle(f"Angular scale decomposition — {FILES[fi].name} ({freq:.1f} MHz, z={1420.405/freq - 1:.1f})",
                 fontsize=14, y=1.01)
    plt.show()

# Print power fraction table for all 3 frequencies
print(f"{'Band':<16}", end="")
for fi in freq_indices:
    with h5py.File(FILES[fi], "r") as fl:
        freq = fl["Header"]["freq_array"][0] / 1e6
    print(f"{freq:.1f} MHz  ", end="")
print()
print("-" * 52)

for ell_lo, ell_hi in bands:
    print(f"ell {ell_lo:>4d}–{ell_hi:<4d}   ", end="")
    for fi in freq_indices:
        with h5py.File(FILES[fi], "r") as fl:
            sky_map = fl["Data"]["stokes"][0, 0, :]
        cl = hp.anafast(sky_map, lmax=lmax_decomp)
        total = np.sum(cl[2:])
        band_pwr = np.sum(cl[ell_lo:ell_hi+1])
        print(f"  {band_pwr/total*100:5.1f}%    ", end="")
    print()

## 4. Pixel Value Histogram & Gaussianity Test

Since the EOR model is a Gaussian Random Field, the pixel values at each frequency should follow a Gaussian distribution. We test this with histograms and a Shapiro-Wilk test (on a subsample).

In [ ]:
# Histogram at 3 frequencies + Gaussianity tests
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
indices = [0, len(FILES)//2, len(FILES)-1]

for ax, idx in zip(axes, indices):
    with h5py.File(FILES[idx], "r") as fl:
        sky = fl["Data"]["stokes"][0, 0, :]
        freq = fl["Header"]["freq_array"][0] / 1e6
    
    mean, std = np.mean(sky), np.std(sky)
    
    ax.hist(sky, bins=200, density=True, alpha=0.7, color="steelblue", label="Data")
    
    # Overlay Gaussian fit
    x = np.linspace(sky.min(), sky.max(), 500)
    ax.plot(x, stats.norm.pdf(x, mean, std), "r-", lw=2, label=f"Gaussian\n$\\mu$={mean:.4f}\n$\\sigma$={std:.4f}")
    
    # Shapiro-Wilk on subsample (max 5000 for the test)
    rng = np.random.default_rng(42)
    subsample = rng.choice(sky, size=5000, replace=False)
    stat_sw, p_sw = stats.shapiro(subsample)
    
    # Skewness and kurtosis
    skew = stats.skew(sky)
    kurt = stats.kurtosis(sky)
    
    ax.set_title(f"{freq:.2f} MHz\nskew={skew:.3f}, kurt={kurt:.3f}\nShapiro p={p_sw:.3e}")
    ax.set_xlabel("Stokes I [Jy/sr]")
    ax.legend(fontsize=9)

axes[0].set_ylabel("Probability density")
plt.tight_layout()
plt.show()

print("Interpretation: skewness~0 and excess kurtosis~0 indicate Gaussianity.")

## 5. Angular Power Spectrum C_ell

Compute the spherical harmonic power spectrum at multiple frequencies using `healpy.anafast`. The GRF was generated with P(k) ~ k^{-2.7} and ell_max=1250, so C_ell should follow a declining power law and cut off near ell ~1250. We also check that C_ell is consistent across frequencies.

In [ ]:
# Compute C_ell at several frequencies
nside = 256
lmax = 3 * nside - 1  # 767

sample_indices = np.linspace(0, len(FILES)-1, 6, dtype=int)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

for idx in sample_indices:
    with h5py.File(FILES[idx], "r") as fl:
        sky = fl["Data"]["stokes"][0, 0, :]
        freq = fl["Header"]["freq_array"][0] / 1e6
    
    # Remove monopole before computing power spectrum
    sky_demono = hp.remove_monopole(sky)
    cl = hp.anafast(sky_demono, lmax=lmax)
    ell = np.arange(len(cl))
    
    ax1.loglog(ell[1:], cl[1:], alpha=0.7, label=f"{freq:.1f} MHz")
    ax2.loglog(ell[1:], ell[1:] * (ell[1:] + 1) * cl[1:] / (2 * np.pi), alpha=0.7, label=f"{freq:.1f} MHz")
    norm_ps = ell[1:] * (ell[1:] + 1) * cl[1:] / (2 * np.pi)
    ind_max = np.argmax(norm_ps)
    print(f"Freq: {freq:.1f} MHz, Peak Cl = {ell[1:][ind_max]:.1f}, peak power={norm_ps[ind_max]:.1e}")

# Mark ell_max from covariance generation
for ax in (ax1, ax2):
    ax.axvline(1250, color="k", ls="--", alpha=0.5, label="ell_max=1250 (input)")
    ax.set_xlabel(r"Multipole $\ell$")
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    # ax.set_ylim(1e-2, 1e-1)  # Set y-limits for better visibility
    # ax.set_xlim(1e2, 1e3)  # Set x-limits for better visibility
    ax.axvline(200)
    ax.axvline(350)

ax1.set_ylabel(r"$C_\ell$ [(Jy/sr)$^2$]")
ax1.set_title(r"Angular Power Spectrum $C_\ell$")

ax2.set_ylabel(r"$\ell(\ell+1) C_\ell / 2\pi$ [(Jy/sr)$^2$]")
ax2.set_title(r"$D_\ell$ (dimensionless power)")

plt.tight_layout()
plt.show()

# Fit power-law slope to C_ell in the range ell=10..500
ell_fit = np.arange(10, 501)
cl_ref = hp.anafast(hp.remove_monopole(test_map), lmax=lmax)
slope, intercept = np.polyfit(np.log10(ell_fit), np.log10(cl_ref[10:501]), 1)
print(f"C_ell power-law slope (ell=10..500): {slope:.3f}  (input P(k)~k^-2.7)")


In [ ]:
print(np.rad2deg(hp.nside2resol(256, arcmin=False)) ) 

## 6. Cross-frequency Angular Power Spectrum C_ell(nu1, nu2)

Compute the cross-power spectrum between frequency pairs. Adjacent channels should be highly correlated; distant channels less so. This tests that the cross-frequency covariance structure from the GRF generator is present.

In [ ]:
# Cross-power spectrum: first channel vs channels at increasing separation
with h5py.File(FILES[0], "r") as fl:
    map_ref = hp.remove_monopole(fl["Data"]["stokes"][0, 0, :])
    freq_ref = fl["Header"]["freq_array"][0] / 1e6

separations = [1, 5, 10, 20, 44, 88]  # channel offsets
fig, ax = plt.subplots(figsize=(10, 5))

cl_auto = hp.anafast(map_ref, lmax=lmax)
ell = np.arange(len(cl_auto))
ax.loglog(ell[2:], cl_auto[2:], "k-", lw=2, label=f"Auto: {freq_ref:.1f} MHz")

for sep in separations:
    if sep >= len(FILES):
        continue
    with h5py.File(FILES[sep], "r") as fl:
        map_cross = hp.remove_monopole(fl["Data"]["stokes"][0, 0, :])
        freq_cross = fl["Header"]["freq_array"][0] / 1e6
    
    cl_cross = hp.anafast(map_ref, map_cross, lmax=lmax)
    delta_freq = freq_cross - freq_ref
    ax.loglog(ell[2:], np.abs(cl_cross[2:]), alpha=0.7, 
              label=f"Cross: +{sep}ch ({delta_freq:.2f} MHz)")

ax.axvline(1250, color="k", ls="--", alpha=0.4)
ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"|$C_\ell$| [(Jy/sr)$^2$]")
ax.set_title("Cross-frequency angular power spectrum")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Build Frequency Data Cube

Load all channels into a (Nfreq, Npix_sample) array for spectral and cross-frequency analysis. We subsample pixels to keep memory manageable.

In [ ]:
# Build data cube: (Nfreq, Npix_sample)
# Sample every 100th pixel for efficiency + keep full maps for variance calculation
PIX_STRIDE = 100
npix_total = 786432
npix_sample = len(np.arange(0, npix_total, PIX_STRIDE))  # correct: ceil(786432/100) = 7865

cube = np.zeros((len(FILES), npix_sample))
means = np.zeros(len(FILES))
stds = np.zeros(len(FILES))

for i, f in enumerate(FILES):
    with h5py.File(f, "r") as fl:
        full_map = fl["Data"]["stokes"][0, 0, :]
        cube[i, :] = full_map[::PIX_STRIDE]
        means[i] = np.mean(full_map)
        stds[i] = np.std(full_map)

print(f"Data cube shape: {cube.shape} (Nfreq, Npix_sample)")
print(f"Memory: {cube.nbytes / 1e6:.1f} MB")

## 8. Frequency Spectra (Per-Pixel)

Plot the Stokes I value as a function of frequency for several individual pixels. The EOR signal should show spectral structure (not a smooth power law like foregrounds).

In [ ]:
# Per-pixel frequency spectra
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Plot individual pixel spectra
rng = np.random.default_rng(42)
pix_sample = rng.choice(npix_sample, 8, replace=False)
for p in pix_sample:
    ax1.plot(freqs_mhz, cube[:, p], alpha=0.6, label=f"pix {p*PIX_STRIDE}")

ax1.set_xlabel("Frequency [MHz]")
ax1.set_ylabel("Stokes I [Jy/sr]")
ax1.set_title("Individual pixel spectra")
ax1.legend(fontsize=7, ncol=2)
ax1.grid(True, alpha=0.3)

# Mean spectrum across all sampled pixels
mean_spec = np.mean(cube, axis=1)
std_spec = np.std(cube, axis=1)

ax2.plot(freqs_mhz, mean_spec, "k-", lw=2, label="Mean")
ax2.fill_between(freqs_mhz, mean_spec - std_spec, mean_spec + std_spec, 
                  alpha=0.3, color="steelblue", label=r"$\pm 1\sigma$")
ax2.set_xlabel("Frequency [MHz]")
ax2.set_ylabel("Stokes I [Jy/sr]")
ax2.set_title("Mean spectrum (all sampled pixels)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Cross-Frequency Correlation Matrix

Compute the Pearson correlation between all pairs of frequency channels. The EOR signal should decorrelate with increasing frequency separation (off-diagonal falls off). Foregrounds would show near-unity correlation everywhere.

In [ ]:
# Cross-frequency correlation matrix
# Mean-subtract each channel before computing correlations to avoid
# the monopole dominating the correlation structure
cube_centered = cube - np.mean(cube, axis=1, keepdims=True)
corr_matrix = np.corrcoef(cube_centered)  # (Nfreq, Nfreq)
np.fill_diagonal(corr_matrix, 1.0)  # ensure exact 1 on diagonal

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Full correlation matrix
im = ax1.imshow(corr_matrix, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1,
                extent=[freqs_mhz[0], freqs_mhz[-1], freqs_mhz[-1], freqs_mhz[0]])
plt.colorbar(im, ax=ax1, label="Pearson r")
ax1.set_xlabel("Frequency [MHz]")
ax1.set_ylabel("Frequency [MHz]")
ax1.set_title("Cross-frequency correlation matrix")

# Correlation as a function of frequency lag
n_freq = len(freqs_mhz)
delta_freq_mhz = np.diff(freqs_mhz)[0]
max_lag = n_freq // 2
lag_mhz = np.arange(max_lag) * delta_freq_mhz
corr_vs_lag = np.zeros(max_lag)

for lag in range(max_lag):
    diag_vals = np.diag(corr_matrix, k=lag)
    corr_vs_lag[lag] = np.nanmean(diag_vals)

ax2.plot(lag_mhz, corr_vs_lag, "o-", markersize=3)
ax2.axhline(0, color="k", ls="--", alpha=0.3)
ax2.set_xlabel("Frequency separation [MHz]")
ax2.set_ylabel("Mean correlation")
ax2.set_title("Decorrelation with frequency separation")
ax2.grid(True, alpha=0.3)

# Estimate decorrelation bandwidth (where correlation drops to 1/e)
e_fold = 1 / np.e
cross_idx = np.argmax(corr_vs_lag < e_fold) if np.any(corr_vs_lag < e_fold) else -1
if cross_idx > 0:
    print(f"Decorrelation bandwidth (r < 1/e): ~{lag_mhz[cross_idx]:.2f} MHz")
else:
    print(f"Signal remains correlated across full bandwidth ({lag_mhz[-1]:.2f} MHz)")

plt.tight_layout()
plt.show()

## 10. Delay Spectrum / Line-of-Sight Power Spectrum

FFT along the frequency axis to compute the delay (tau) power spectrum. This is the line-of-sight analog of P(k_parallel). A Blackman window is applied to reduce spectral leakage. We also convert delay to cosmological k_parallel using Planck15 cosmology and the standard HERA convention (Parsons et al. 2012).

In [ ]:
# Delay spectrum along the frequency axis
delta_nu = np.diff(freqs_hz)[0]  # Hz
n_freq = len(freqs_hz)

# Blackman window to reduce spectral leakage
window = np.blackman(n_freq)
window /= np.sqrt(np.mean(window**2))  # normalize to preserve power

# Compute delay spectrum for each sampled pixel
delay_ps = np.zeros((npix_sample, n_freq))
for p in range(npix_sample):
    spec = cube[:, p] - np.mean(cube[:, p])  # remove mean (DC / zero-delay)
    ft = np.fft.fft(spec * window)
    delay_ps[p, :] = np.abs(ft)**2 / n_freq  # normalize by N (Parseval-consistent)

# Average over pixels
mean_delay_ps = np.mean(delay_ps, axis=0)
mean_delay_ps = np.fft.fftshift(mean_delay_ps)

# Delay axis
tau = np.fft.fftshift(np.fft.fftfreq(n_freq, delta_nu))  # seconds
tau_ns = tau * 1e9  # nanoseconds

# Convert delay to k_parallel using cosmology (Parsons et al. 2012 convention)
# k_parallel = 2*pi*tau * H(z) * f_21 / (c * (1+z)^2)
# Use consistent units: H(z) in km/s/Mpc, c in km/s -> k_parallel in 1/Mpc
cosmo = cosmology.Planck15
f_21 = 1420.405e6  # Hz
z_center = f_21 / np.mean(freqs_hz) - 1
H_z = cosmo.H(z_center).value  # km/s/Mpc
c_kms = 299792.458  # km/s

k_parallel = 2 * np.pi * np.abs(tau) * H_z * f_21 / (c_kms * (1 + z_center)**2)  # 1/Mpc

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Delay spectrum
ax1.semilogy(tau_ns, mean_delay_ps, "b-", alpha=0.8)
ax1.set_xlabel(r"Delay $\tau$ [ns]")
ax1.set_ylabel(r"$|\tilde{I}(\tau)|^2 / N$ [(Jy/sr)$^2$]")
ax1.set_title("Delay spectrum (pixel-averaged)")
ax1.grid(True, alpha=0.3)

# k_parallel power spectrum (positive delays only)
pos_mask = tau > 0
k_par_pos = k_parallel[pos_mask]
ps_pos = mean_delay_ps[pos_mask]

ax2.loglog(k_par_pos, ps_pos, "b-", alpha=0.8)
ax2.set_xlabel(r"$k_\parallel$ [Mpc$^{-1}$]")
ax2.set_ylabel(r"P($k_\parallel$) [(Jy/sr)$^2$]")
ax2.set_title(f"Line-of-sight power spectrum (z={z_center:.1f})")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Center redshift: z = {z_center:.2f}")
print(f"H(z) = {H_z:.1f} km/s/Mpc")
print(f"Delay resolution: {1/(n_freq*delta_nu)*1e9:.2f} ns")
print(f"Max delay: {1/(2*delta_nu)*1e9:.2f} ns")
print(f"k_parallel range: {k_par_pos.min():.4f} – {k_par_pos.max():.4f} Mpc^-1")

## 11. Variance & Mean vs Frequency

The variance of pixel values as a function of frequency is a key EOR diagnostic. For a GRF with P(k) ~ k^{-2.7}, the variance should increase with frequency (as redshift decreases and the signal grows). The mean should be relatively stable if the monopole is set consistently.

In [ ]:
# Variance and mean vs frequency (from full maps computed earlier)
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4.5))

ax1.plot(freqs_mhz, means, "o-", markersize=3, color="navy")
ax1.set_xlabel("Frequency [MHz]")
ax1.set_ylabel("Mean Stokes I [Jy/sr]")
ax1.set_title("Mean vs Frequency")
ax1.grid(True, alpha=0.3)

ax2.plot(freqs_mhz, stds**2, "o-", markersize=3, color="darkred")
ax2.set_xlabel("Frequency [MHz]")
ax2.set_ylabel("Variance [(Jy/sr)$^2$]")
ax2.set_title("Variance vs Frequency")
ax2.grid(True, alpha=0.3)

# Also plot std/mean (coefficient of variation) - should change smoothly
ax3.plot(freqs_mhz, stds / means, "o-", markersize=3, color="darkgreen")
ax3.set_xlabel("Frequency [MHz]")
ax3.set_ylabel(r"$\sigma / \mu$ (CV)")
ax3.set_title("Coefficient of Variation vs Frequency")
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Check for any outlier channels
cv = stds / means
cv_median = np.median(cv)
cv_mad = np.median(np.abs(cv - cv_median))
outliers = np.where(np.abs(cv - cv_median) > 5 * cv_mad)[0]
if len(outliers) > 0:
    print(f"WARNING: {len(outliers)} outlier channel(s) detected: {[FILES[i].name for i in outliers]}")
else:
    print("No outlier channels detected (CV within 5*MAD of median).")

## 12. Frequency-Pixel Waterfall Plot

2D image of (frequency x pixel) to visually inspect spectral and spatial structure simultaneously. This is useful for spotting artifacts like banding or discontinuities.

In [ ]:
# Waterfall plot: frequency vs pixel index
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Raw waterfall
im1 = ax1.imshow(cube, aspect="auto", origin="lower", cmap="inferno",
                  extent=[0, npix_sample, freqs_mhz[0], freqs_mhz[-1]])
plt.colorbar(im1, ax=ax1, label="Jy/sr")
ax1.set_xlabel("Pixel index (sampled)")
ax1.set_ylabel("Frequency [MHz]")
ax1.set_title("Stokes I waterfall")

# Mean-subtracted waterfall (highlights spectral structure)
cube_detrended = cube - means[:, None]
vmax = np.percentile(np.abs(cube_detrended), 99)
im2 = ax2.imshow(cube_detrended, aspect="auto", origin="lower", cmap="RdBu_r",
                  vmin=-vmax, vmax=vmax,
                  extent=[0, npix_sample, freqs_mhz[0], freqs_mhz[-1]])
plt.colorbar(im2, ax=ax2, label="Jy/sr")
ax2.set_xlabel("Pixel index (sampled)")
ax2.set_ylabel("Frequency [MHz]")
ax2.set_title("Mean-subtracted waterfall")

plt.tight_layout()
plt.show()

## 13. Stokes Q/U/V Check

The EOR GRF model should be unpolarized (only Stokes I non-zero). Verify that Q, U, V are identically zero.

In [ ]:
# Check Stokes Q, U, V are zero
stokes_labels = ["I", "Q", "U", "V"]
check_files = [FILES[0], FILES[len(FILES)//2], FILES[-1]]

print("Stokes parameter max absolute values:")
print(f"{'File':<16} {'I max':>12} {'Q max':>12} {'U max':>12} {'V max':>12}")
print("-" * 68)
for f in check_files:
    with h5py.File(f, "r") as fl:
        stokes = fl["Data"]["stokes"][:]
    maxvals = [np.max(np.abs(stokes[s, 0, :])) for s in range(4)]
    print(f"{f.name:<16} {maxvals[0]:12.6e} {maxvals[1]:12.6e} {maxvals[2]:12.6e} {maxvals[3]:12.6e}")

quv_nonzero = any(maxvals[s] > 0 for s in range(1, 4))
if quv_nonzero:
    print("\nWARNING: Non-zero polarization detected!")
else:
    print("\nOK: Q, U, V are all exactly zero (unpolarized, as expected).")

## 14. Summary Statistics Table

In [ ]:
# Summary statistics
print("=" * 80)
print("EOR GRF Sky Model Diagnostic Summary")
print("=" * 80)
print(f"Sky model path:  {SKY_DIR}")
print(f"Number of files: {len(FILES)}")
print(f"Channel range:   {FILES[0].name} – {FILES[-1].name}")
print(f"Freq range:      {freqs_mhz[0]:.4f} – {freqs_mhz[-1]:.4f} MHz")
print(f"Redshift range:  z = {z_arr[-1]:.3f} – {z_arr[0]:.3f}")
print(f"Channel spacing: {delta_nu/1e3:.2f} kHz")
print(f"Total bandwidth: {(freqs_mhz[-1] - freqs_mhz[0]):.4f} MHz")
print(f"HEALPix nside:   {nsides[0]}")
print(f"Pixel ordering:  {list(set(orderings))}")
print(f"Npix per map:    {stokes_shapes[0][2]}")
print(f"Pixel area:      {hp.nside2pixarea(nsides[0], degrees=True)*3600:.2f} arcmin^2")
print()
print("Per-channel statistics (across all 89 channels):")
print(f"  Mean Stokes I: {np.mean(means):.6f} +/- {np.std(means):.6f} Jy/sr")
print(f"  Std Stokes I:  {np.mean(stds):.6f} (range {np.min(stds):.6f} – {np.max(stds):.6f}) Jy/sr")
print(f"  Min pixel:     {np.min(cube):.6f} Jy/sr")
print(f"  Max pixel:     {np.max(cube):.6f} Jy/sr")
print(f"  All values >0: {np.all(cube > 0)}")
print("=" * 80)